# Experiment 009: the 1.5B cell, seeds 0 and 1 (Colab)

The missing middle point of the rq2 scale curve. Frozen protocol v1, a100,
condition sft_grpo, seeds 0 and 1 in one sweep. Base evals are shared across
the seeds inside the experiment, so seed 0 pays for them once and seed 1 skips.

Realistic budget: ~12-14h for both seeds, so this probably spans two colab
sessions. That is fine, every stage is resume-safe: reconnect, run all again,
finished stages skip in seconds and training resumes from the last checkpoint.

## 0. Compute units and mount Drive

Fill `cu_before` now. The open record is saved straight after Drive mounts,
so a disconnected session can be closed when the next runtime starts.

In [1]:
# note the balance before anything burns units, then save it after drive mounts
import glob, json, os, time
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 350.18   # e.g. 87.3
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

# close unfinished session with the balance captured at this runtime start
old_sessions = [p for p in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not p.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None:
    print('reminder: fill cu_before, then rerun this cell to save the session')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])

    if not os.path.exists(open_path):
        start = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                 'notebook': '009_colab_1p5b_full', 'colab_cu_before': cu_before}
        with open(open_path, 'w') as f:
            json.dump(start, f, indent=2)
        print('saved open session:', session_id)
print('drive mounted ->', ROOT)


session 20260720_213054 | cu_before = 350.18
Mounted at /content/drive
closed previous session: 20260720_211926
saved open session: 20260720_213054
drive mounted -> /content/drive/MyDrive/inm363


## 1. Check the GPU

In [2]:
import subprocess
# reportable cells run on a100/bf16 only (runtime consistency is part of the
# protocol), so stop here instead of finding out five hours into training
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpu.strip() or 'no gpu visible')
assert 'A100' in gpu, 'wrong runtime: pick a100 under runtime > change runtime type'

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-f2f64550-d0de-1957-888a-d71421352bd7)


## 2. Clone or update the repo (automatic)

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    # a resumed runtime already has the clone, in that case pull instead of failing
    # and silently running whatever old code is lying around
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 1541, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 1541 (delta 7), reused 14 (delta 7), pack-reused 1521 (from 1)
Receiving objects: 100% (1541/1541), 47.27 MiB | 19.96 MiB/s, done.
Resolving deltas: 100% (914/914), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

In [4]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 154.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.3 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Note for a fresh notebook: colab secrets are granted per notebook, so open the
key icon in the left sidebar and switch on notebook access for `WANDB_API_KEY` before running this, otherwise wandb prompts interactively
halfway into training. This key is only used for experiment tracking.


In [5]:
import getpass
import os

wkey = None
try:
    from google.colab import userdata
    wkey = userdata.get('WANDB_API_KEY')
except Exception:
    wkey = None

if not wkey:
    wkey = getpass.getpass('W&B API key: ').strip()
if not wkey:
    raise RuntimeError('WANDB_API_KEY is required')

os.environ['WANDB_API_KEY'] = wkey
with open('wandb_api_key.txt', 'w') as f:
    f.write(wkey.strip())
print('W&B key loaded:', os.path.exists('wandb_api_key.txt'))


W&B key loaded: True


## 5. Check the Drive record

In [6]:
if os.path.exists(open_path):
    print('open compute-unit record:', open_path)
else:
    print('open record missing, rerun section 0 with cu_before filled')


open compute-unit record: /content/drive/MyDrive/inm363/results/open_cu_sessions/20260720_213054.json


## 6. Build the data splits

In [7]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 23.6MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.68MB/s,  224kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.68MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 320kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 320kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 399740.26 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 317764.90 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 81.1MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 139MB/s]

high_school_psychology/test-00000-of-000(…): downloading bytes:   0% 0.00/92.8k [00:00<?, ?B

## 7. Wiring check

In [8]:
!python scripts/run.py --config configs/009_colab_full_b15_seed0.yaml

env=COLAB mode=FULL scale=B15 condition=SFT_GRPO
model=Qwen/Qwen2.5-1.5B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/009_COLAB_FULL_B15_sft_grpo_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/009_COLAB_FULL_B15_sft_grpo_seed0/
local smoke wiring ok


## 8. Full runs, both seeds

**If colab disconnects:** re-run this cell, finished stages skip.

In [ ]:
# Python loop with ! calls so the logs stream live. The exit code check stops
# the sweep if a seed fails, otherwise the next seed would burn the session
# while the failure sits unnoticed
for cfg in ('009_colab_full_b15_seed0', '009_colab_full_b15_seed1'):
    print(f'===== {cfg} =====')
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/run.py --config configs/{cfg}.yaml --stage all
    if _exit_code != 0:
        raise SystemExit(f'{cfg} failed with exit code {_exit_code}, fix that before the next seed runs')

Streaming output truncated to the last 5000 lines.
│ │ due to their  │ sanctions     │               │              │           │ │
│ │ ability to    │ against       │               │              │           │ │
│ │ threaten      │ punitive      │               │              │           │ │
│ │ governments,  │ corporations. │               │              │           │ │
│ │ in the face   │ In turn, from │               │              │           │ │
│ │ of ________   │ fighting for  │               │              │           │ │
│ │ with          │ the rights of │               │              │           │ │
│ │ relocation to │ impoverished  │               │              │           │ │
│ │ other         │ villagers to  │               │              │           │ │
│ │ territories,  │ opposing      │               │              │           │ │
│ │ which Beck    │ US-led wars,  │               │              │           │ │
│ │ (1998)        │ these         │               │       

## 9. Peek at the results

In [ ]:
import json, os
D = '/content/drive/MyDrive/inm363'
for s in (0, 1):
    d = os.path.join(D, 'results', f'009_COLAB_FULL_B15_sft_grpo_seed{s}')
    sm = os.path.join(d, 'summary.json')
    if os.path.exists(sm):
        j = json.load(open(sm))
        print(f"seed{s}: base {j['base_acc']}  sft {j['sft_acc']}  grpo {j['grpo_acc']}")
    else:
        print(f'seed{s}: no summary yet ({d})')

seed0: base {'easy': 0.4311717861205916, 'medium': 0.3497536945812808, 'hard': 0.2702020202020202}  sft {'easy': 0.6382252559726962, 'medium': 0.5763546798029556, 'hard': 0.30303030303030304}  grpo {'easy': 0.8088737201365188, 'medium': 0.6272577996715928, 'hard': 0.44696969696969696}
seed1: base {'easy': 0.4311717861205916, 'medium': 0.3497536945812808, 'hard': 0.2702020202020202}  sft {'easy': 0.6279863481228669, 'medium': 0.5320197044334976, 'hard': 0.29545454545454547}  grpo {'easy': 0.7679180887372014, 'medium': 0.5977011494252874, 'hard': 0.39646464646464646}


## 10. Close the units record

In [ ]:
# closing balance, completes the open record for this session
import json, os, time
cu_after = 276.19    # fill from the resources panel again
if cu_after is None:
    print('reminder: fill cu_after for the record')
elif not os.path.isdir('/content/drive/MyDrive'):
    print('drive not mounted, run the mount cell first')
else:
    os.makedirs(OPEN_UNITS, exist_ok=True)
    open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
    if not os.path.exists(open_path):
        print('no open record for this session, it may already be closed')
    else:
        with open(open_path) as f:
            rec = json.load(f)
        rec['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        rec['colab_cu_after'] = cu_after
        rec['colab_cu_used'] = round(rec['colab_cu_before'] - cu_after, 2)
        rec['closed_on_resume'] = False
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(rec) + '\n')
        os.remove(open_path)
        print('logged:', rec)

reminder: fill cu_after for the record
